**Import Libraries**

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

**Initialize MediaPipe Hands**

In [6]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print("Libraries imported successfully!")

Libraries imported successfully!


2. Understanding MediaPipe Hands

 MediaPipe Hands detects **21 landmarks** per hand:
 
 - **0:** Wrist
 - **1-4:** Thumb (CMC, MCP, IP, Tip)
 - **5-8:** Index finger
 - **9-12:** Middle finger
 - **13-16:** Ring finger
 - **17-20:** Pinky finger

**Test on Static Image**

In [12]:
import cv2
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)

print("Press q qto quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

    cv2.imshow("Hand Detection Test", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
hands.close()
cv2.destroyAllWindows()

I0000 00:00:1766724455.837603       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2


Press q qto quit


Extracting Landmark Coordinates

In [14]:
def extract_landmarks(hand_landmarks, image_shape):
 
    h, w, _ = image_shape
    landmarks = []
    
    for landmark in hand_landmarks.landmark:
        # Convert normalized coordinates to pixel coordinates
        x = int(landmark.x * w)
        y = int(landmark.y * h)
        landmarks.append([x, y])
    
    return np.array(landmarks)

# Test extraction on our captured image
if results.multi_hand_landmarks:
    for idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
        landmarks_array = extract_landmarks(hand_landmarks, test_image.shape)
        print(f"\n Hand {idx + 1} Landmarks Shape: {landmarks_array.shape}")
        print(f"First 5 landmarks (x, y):\n{landmarks_array[:5]}")

**Real-Time Webcam Detection**

In [17]:
def run_hand_detection(duration_seconds=10):
 
    cap = cv2.VideoCapture(0)
    
    # Initialize MediaPipe Hands for video
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    print(" Starting webcam Press 'q' to quit")
    
    import time
    start_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Checking duration
        if time.time() - start_time > duration_seconds:
            break
        
        # Flip frame horizontally for mirror view
        frame = cv2.flip(frame, 1)
        
        # Convert to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Process frame
        results = hands.process(rgb_frame)
        
        # Drawing landmarks
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )
        
        # Displaying frame
        cv2.imshow('Hand Detection', frame)
        
        # Quit on 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    print(" Webcam stopped")

Run detection for 10 seconds
 Uncomment the line below to test
 run_hand_detection(duration_seconds=10)

Key Takeaways
 
 1. **MediaPipe Hands** detects 21 3D landmarks per hand
 2. **Landmark coordinates** are normalized (0-1 range)
 3. **Two modes:**
    - `static_image_mode=True` → For images
    - `static_image_mode=False` → For video (faster tracking)
 4. **Next step:** Extract features from these landmarks for gesture classification